# Tuần 02: Dữ liệu nghiên cứu dưới dạng bảng

Tuần này bạn học cách nhìn một dataset như một thiết kế nghiên cứu: một dòng là gì, mỗi cột ghi gì, và bảng đó sẽ hỗ trợ paper như thế nào.

## Mục tiêu học tập

Cuối notebook này, bạn có thể:

1. giải thích row, column, value và schema;
2. đọc CSV nhỏ bằng `csv.DictReader`;
3. chọn cột bắt buộc và cột tùy chọn;
4. viết data description 120-160 từ.

## Cách bắt đầu

1. Đọc bản HTML này trước.
2. Mở Colab nếu muốn chạy notebook trong browser.
3. Chạy cell từ trên xuống.
4. Chỉ sửa cell có ghi **Bạn sửa**.
5. Không thêm dữ liệu định danh người thật.

## 1. Ý tưởng chính: một dòng là gì?

Một dataset tốt cần trả lời được câu hỏi: **mỗi dòng đại diện cho điều gì?**

- Row: một đơn vị quan sát.
- Column: một biến hoặc thuộc tính.
- Value: giá trị trong một ô.
- Schema: danh sách cột và ý nghĩa của chúng.

## 2. Đọc CSV thiết kế bảng

**Code mẫu - chỉ cần chạy:** cell tiếp theo tìm CSV Week 2. Nếu dùng Colab và chưa có file local, Python sẽ tải bản public từ GitHub.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import csv

possible_paths = [
    Path("data/raw/week02_research_table_examples.csv"),
    Path("weeks/week-02-research-data-tables/data/raw/week02_research_table_examples.csv"),
]

data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is None:
    week_dir = Path(".")
    data_dir = week_dir / "data/raw"
    data_dir.mkdir(parents=True, exist_ok=True)
    data_path = data_dir / "week02_research_table_examples.csv"
    source_url = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/main/weeks/week-02-research-data-tables/data/raw/week02_research_table_examples.csv"
    try:
        urlretrieve(source_url, data_path)
    except Exception as error:
        raise RuntimeError(
            "Python could not download the Week 02 CSV. "
            "Please upload week02_research_table_examples.csv to data/raw/ and run this cell again."
        ) from error
else:
    week_dir = data_path.parents[2]

print("Data file exists:", data_path.exists())
print("Data file path:", data_path)

Data file exists: True
Data file path: weeks/week-02-research-data-tables/data/raw/week02_research_table_examples.csv


In [2]:
with data_path.open(encoding="utf-8") as file:
    rows = list(csv.DictReader(file))

print("Number of table plans:", len(rows))
print("Column names:", list(rows[0].keys()))

Number of table plans: 4
Column names: ['track_id', 'track', 'data_unit', 'starter_table', 'required_columns', 'optional_columns', 'paper_output', 'beginner_task']


## 3. Xem schema theo từng track

**Code mẫu - chỉ cần chạy:** đọc output và chú ý ba cột: `data_unit`, `required_columns`, `paper_output`.

In [3]:
for row in rows:
    print(row["track_id"], "-", row["track"])
    print("  One row means:", row["data_unit"])
    print("  Required columns:", row["required_columns"])
    print("  Paper output:", row["paper_output"])
    print()

TCSOL_SHORT - Short-term Chinese teaching
  One row means: learner pre/post score record
  Required columns: learner_id,group,pre_score,post_score,focus,activity_date
  Paper output: clean data template and source note

CONTRASTIVE - Chinese-Vietnamese contrastive analysis
  One row means: Chinese-Vietnamese grammar example with teaching note
  Required columns: example_id,zh_sentence,vi_equivalent,feature,contrast_point,teaching_note
  Paper output: contrastive example table

MTPE - Machine translation and MTPE
  One row means: translation segment with human post-editing and simplified error label
  Required columns: segment_id,zh_source,vi_mt,vi_postedit,local_error_label,severity
  Paper output: simplified error-label table

POLICY - Education policy
  One row means: manually coded policy excerpt
  Required columns: doc_id,title,issuing_body,date,excerpt,theme_code
  Paper output: theme count table or source matrix



## 4. Chọn một table plan

**Bạn sửa:** chọn một ID và chỉ sửa giá trị của `selected_track_id`.

- `TCSOL_SHORT`
- `CONTRASTIVE`
- `MTPE`
- `POLICY`

In [4]:
selected_track_id = "POLICY"

valid_track_ids = [row["track_id"] for row in rows]
if selected_track_id not in valid_track_ids:
    raise ValueError("Please choose one of these track IDs: " + ", ".join(valid_track_ids))

selected = next(row for row in rows if row["track_id"] == selected_track_id)

print("Selected track:", selected["track"])
print("One row means:", selected["data_unit"])
print("Required columns:", selected["required_columns"])
print("Optional columns:", selected["optional_columns"])

Selected track: Education policy
One row means: manually coded policy excerpt
Required columns: doc_id,title,issuing_body,date,excerpt,theme_code
Optional columns: policy_level,url,access_date,coding_note


## 5. Biến cột thành danh sách

**Code mẫu - chỉ cần chạy:** Python tách chuỗi cột bằng dấu phẩy thành list. Tuần này bạn chỉ cần đọc output.

In [5]:
required_columns = [column.strip() for column in selected["required_columns"].split(",")]
optional_columns = [column.strip() for column in selected["optional_columns"].split(",")]

print("Required column count:", len(required_columns))
print("Required columns:")
for column in required_columns:
    print("-", column)

print("Optional columns:")
for column in optional_columns:
    print("-", column)

Required column count: 6
Required columns:
- doc_id
- title
- issuing_body
- date
- excerpt
- theme_code
Optional columns:
- policy_level
- url
- access_date
- coding_note


## 6. Xuất table plan nhỏ

**Code mẫu - chỉ cần chạy:** bảng này là sản phẩm nhỏ của Week 2. Nó ghi track, row unit, columns, output và privacy risk.

In [6]:
privacy_risk_by_track = {
    "TCSOL_SHORT": "medium: learner data must be anonymized",
    "CONTRASTIVE": "low: example data, but cite sources",
    "MTPE": "low to medium: avoid private client texts",
    "POLICY": "low: public documents, but keep URL and access date",
}

summary = {
    "selected_track": selected["track"],
    "one_row_means": selected["data_unit"],
    "required_columns": "; ".join(required_columns),
    "optional_columns": "; ".join(optional_columns),
    "paper_output": selected["paper_output"],
    "privacy_risk": privacy_risk_by_track[selected_track_id],
}

for key, value in summary.items():
    print(key + ":", value)

selected_track: Education policy
one_row_means: manually coded policy excerpt
required_columns: doc_id; title; issuing_body; date; excerpt; theme_code
optional_columns: policy_level; url; access_date; coding_note
paper_output: theme count table or source matrix
privacy_risk: low: public documents, but keep URL and access date


In [7]:
output_dir = week_dir / "outputs/tables"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "week02_selected_table_plan.csv"

with output_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(summary.keys()), lineterminator="\n")
    writer.writeheader()
    writer.writerow(summary)

print("Saved table plan to:", output_path)

Saved table plan to: weeks/week-02-research-data-tables/outputs/tables/week02_selected_table_plan.csv


## Ghi chú nguồn và quyền riêng tư

- Source: instructor-created Week 2 teaching dataset.
- Access date: 2026-06-03.
- Full table-design dataset: N = 4 track plans.
- Selected table plan: N = 1 selected track.
- Privacy: nếu dùng dữ liệu người học thật, thay tên thật bằng anonymous ID và không public dữ liệu định danh.

## 7. Paper-facing data description

Dùng template này cho bài nộp Week 2. Viết 120-160 từ.

Sentence frame:

> The dataset is organized at the level of [unit of observation]. Each row contains [required columns], which allows the study to [paper purpose].

Điền phần dưới đây:

- Chosen track:
- Small research question:
- One row means:
- Required columns:
- Optional columns:
- Source note:
- Privacy risk:
- Paper-facing data description:

**Đoạn viết của bạn:**

...